In [ ]:
with open('input.txt','r',encoding='utf-8') as f:
    text = f.read()

In [ ]:
print(text[:1000])

In [ ]:
#tokenizer (encode and decode) -> character tokenizer

chars = sorted(list(set(text)))

stoi = {}
for i, ch in enumerate(chars):
    stoi[ch] = i

# number -> char
itos = {}
for i, ch in enumerate(chars):
    itos[i] = ch


def encode(s):
    result = []

    for ch in s:
        result.append(stoi[ch])

    return result


def decode(nums):
    text = ""

    for n in nums:
        text += itos[n]

    return text


encoded = encode("hi")
print(encoded)

decoded = decode(encoded)
print(decoded)

In [ ]:
#implement tiktoken tokenizer
#...

In [ ]:
#encoding entire dataset 

import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data[:1000])

In [ ]:
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

In [ ]:
#creating batches
torch.manual_seed(1337)
block_size = 8
batch_size = 4

def get_batch(split):
  if split == 'train':
    data = train_data
  else:
    data = val_data
  
  indexes = torch.randint(len(data) - block_size,(batch_size,))
  x = torch.stack([data[index:index+block_size] for index in indexes])
  y = torch.stack([data[index+1:index+block_size+1] for index in indexes])
  return x,y

xb, yb = get_batch("train")
print(xb.shape)
print("Inputs : ")
print(xb)
print(yb.shape)
print("Outputs : ")
print(yb)


In [ ]:
vocab_size

In [ ]:
#NN (Bigram model is a one character prediction lnaguage model)
# Bigram model learns:
# current token -> next token probabilities using an embedding lookup table.

# Input shape (B,T) becomes (B,T,C),
# where each token returns C=vocab_size next-token logits.

import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)


  def forward(self,idx,targets=None):
    logits = self.token_embedding_table(idx)

    #pytorch cross_entropy accepts in (X x y) format as we have 3d we are converting into 2d
    
    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits,targets)

    return logits, loss

  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      logits,loss = self(idx)
      logits = logits[:,-1,:] #i want last one's logit
      probabilities = F.softmax(logits,dim=-1)
      idx_new = torch.multinomial(probabilities,num_samples=1) #returns a random idx
      idx = torch.cat((idx,idx_new),dim=1)
    return idx


model = BigramLanguageModel(vocab_size)
logits,loss = model(xb,yb)
print(logits.shape)
print(loss) #around 4.87 as model's initial weights are randomly generated so ln(1/65) = 4.17



In [ ]:
#testing initial random text
initial_idx = torch.tensor([[47]], dtype=torch.long)
encoded = model.generate(initial_idx,max_new_tokens=150)

decoded = decode(encoded[0].tolist())
print(decoded)

